In [1]:
#Parameters
ticker = "AAPL"
start_date = "2024-06-01"
end_date = "2024-11-30"
db_path = "data/data.db"

In [2]:
# Parameters
ticker = "IBM"
start_date = "2024-06-01"
end_date = "2024-11-30"
db_path = "C:\\Users\\shrav\\ucb_mfe\\UCB-MFE-python-preprogram-2026-winter\\Homeworks\\HW2\\Shravani_Koli\\data\\data.db"


In [3]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Connect to SQLite
conn = sqlite3.connect(db_path)
# Load cleaned stock data 
query = """
SELECT ts, open, high, low, close, adj_close, volume, ticker
FROM ohlc_hw_cleaned
WHERE ticker = ?
  AND ts >= ?
  AND ts <= ?
ORDER BY ts
"""

stock_df = pd.read_sql_query(query, conn, params=(ticker, start_date, end_date),parse_dates=["ts"])
conn.close()
print("Ticker:", ticker)
print("Number of records:", len(stock_df))
print("Date range:", start_date, "to", end_date)

stock_df.head()

Ticker: IBM
Number of records: 123
Date range: 2024-06-01 to 2024-11-30


,ts,open,high,low,close,adj_close,volume,ticker
0,2024-06-03,166.54,166.78,163.53,165.28,162.57,2776100,IBM
1,2024-06-04,164.60,166.40,163.88,165.81,163.09,2594203,IBM
2,2024-06-05,166.41,167.79,165.78,167.38,164.63,3049400,IBM
3,2024-06-06,167.38,168.44,166.80,168.20,165.44,2207263,IBM
4,2024-06-07,168.18,171.31,168.06,170.01,167.22,3475500,IBM


In [4]:
#daily logarithmic returns using adjusted closing prices
stock_df["log_return"] = np.log(stock_df["adj_close"] / stock_df["adj_close"].shift(1))

# Calculate 20-day rolling volatility, annualized
stock_df["rolling_volatility"] = (stock_df["log_return"].rolling(window=20).std()* np.sqrt(252))

#cumulative returns
stock_df["cumulative_return"] = (stock_df["adj_close"] / stock_df["adj_close"].iloc[0])
#running peak
stock_df["running_peak"] = stock_df["cumulative_return"].cummax()
#calculate drawdown
stock_df["drawdown"] = (stock_df["cumulative_return"] / stock_df["running_peak"] - 1)

# the calculated columns
stock_df[
    ["ts", "adj_close", "log_return",
     "rolling_volatility", "cumulative_return", "drawdown"]
].head(25)

,ts,adj_close,log_return,rolling_volatility,cumulative_return,drawdown
0,2024-06-03,162.570000,NaN,NaN,1.000000,0.000000
1,2024-06-04,163.090000,0.003194,NaN,1.003199,0.000000
2,2024-06-05,164.630000,0.009398,NaN,1.012671,0.000000
3,2024-06-06,165.440000,0.004908,NaN,1.017654,0.000000
4,2024-06-07,167.220000,0.010702,NaN,1.028603,0.000000
5,2024-06-10,167.580000,0.002151,NaN,1.030817,0.000000
6,2024-06-11,166.540000,-0.006225,NaN,1.024420,-0.006206
7,2024-06-12,166.220000,-0.001923,NaN,1.022452,-0.008116
8,2024-06-13,166.340000,0.000722,NaN,1.023190,-0.007399
9,2024-06-14,166.430000,0.000541,NaN,1.023744,-0.006862


In [5]:
# Using 2% annual risk-free rate
risk_free_rate = 0.02
trading_days = 252
daily_returns = stock_df["log_return"].dropna()

#performance metrics
total_return = (stock_df["adj_close"].iloc[-1] /stock_df["adj_close"].iloc[0] - 1)

annualized_return = np.exp(daily_returns.mean() * trading_days) - 1
annualized_volatility = (daily_returns.std() * np.sqrt(trading_days))

sharpe_ratio = ((daily_returns.mean() * trading_days - risk_free_rate)/ annualized_volatility if annualized_volatility > 0 else np.nan)

max_drawdown = stock_df["drawdown"].min()
best_day = np.exp(daily_returns.max()) - 1
worst_day = np.exp(daily_returns.min()) - 1
positive_days = (daily_returns > 0).mean() * 100

avg_up_day = (np.exp(daily_returns[daily_returns > 0]) - 1).mean()

avg_down_day = (np.exp(daily_returns[daily_returns < 0]) - 1).mean()

#displaying the 10 metrics 
metrics = pd.DataFrame({
    "Metric": [
        "Total Return",
        "Annualized Return",
        "Annualized Volatility",
        "Sharpe Ratio",
        "Maximum Drawdown",
        "Best Day",
        "Worst Day",
        "Positive Days (%)",
        "Average Up Day",
        "Average Down Day"
    ],
    "Value": [
        total_return,
        annualized_return,
        annualized_volatility,
        sharpe_ratio,
        max_drawdown,
        best_day,
        worst_day,
        positive_days,
        avg_up_day,
        avg_down_day
    ]
})

metrics

,Metric,Value
0,Total Return,0.398844
1,Annualized Return,1.000308
2,Annualized Volatility,9.800992
3,Sharpe Ratio,0.068697
4,Maximum Drawdown,-0.957659
5,Best Day,22.996882
6,Worst Day,-0.957279
7,Positive Days (%),61.475410
8,Average Up Day,0.589480
9,Average Down Day,-0.101459


In [6]:
import scrapbook as sb

# Store the metrics so Papermill and Scrapbook can retrieve them
report_metrics = {
    "ticker": ticker,
    "avg_daily_return": float(daily_returns.mean()),
    "avg_volatility": float(stock_df["rolling_volatility"].mean()),
    "max_drawdown": float(max_drawdown),
    "sharpe_ratio": float(sharpe_ratio),
    "total_return": float(total_return)
}

sb.glue("report_metrics", report_metrics)

print("Report metrics saved for:", ticker)
print(report_metrics)

Report metrics saved for: IBM
{'ticker': 'IBM', 'avg_daily_return': 0.00275119571702628, 'avg_volatility': 7.26129910882658, 'max_drawdown': -0.9576591467687533, 'sharpe_ratio': 0.06869726530391773, 'total_return': 0.3988435750753523}


In [7]:
#finding dates with the largest daily price movements
stock_df["daily_return"] = stock_df["adj_close"].pct_change()

extreme_days = stock_df.loc[stock_df["daily_return"].abs().nlargest(10).index,["ts", "adj_close", "daily_return"]]
extreme_days

,ts,adj_close,daily_return
19,2024-07-01,4082.109519,22.996882
13,2024-06-21,1194.664390,5.983891
83,2024-10-04,1457.431847,5.595311
111,2024-11-13,1334.755137,5.330054
86,2024-10-09,1051.274443,3.634636
20,2024-07-02,174.390000,-0.957279
14,2024-06-24,172.140000,-0.855909
84,2024-10-07,225.340000,-0.845386
112,2024-11-14,208.990000,-0.843424
87,2024-10-10,231.200000,-0.780076


In [8]:
#inspecting suspicious AAPL price records
stock_df.loc[
    stock_df["ts"].isin(
        pd.to_datetime([
            "2024-06-27",
            "2024-08-22",
            "2024-09-17"
        ])
    ),
    ["ts", "open", "high", "low", "close", "adj_close"]
]

,ts,open,high,low,close,adj_close
17,2024-06-27,171.12,172.50,170.48,170.85,168.04
55,2024-08-22,197.25,197.92,195.57,195.96,194.43
70,2024-09-17,217.25,218.84,213.00,214.13,212.46
